# ETL Transform: News

This notebook runs the **news ETL pipeline**: ingest from Postgres → transform → save to `financial_news_transformed` → publish to S3.

**Two modes (set `USE_AGENTIC_ONLY` in the next cell):**
- **VADER (default)**: transform with sentiment (VADER), intent, keywords, tickers; then save and S3.
- **Agentic only**: skip VADER; run LLM-based financial metrics extraction only; then save and S3.

**S3 upload modes:**
- **Per-article**: one CSV per article at `news/crypto/[agentic=true|false/]year=.../.../format=csv/{id}.csv`
- **Batch (run)**: one CSV per run at `news/transformed/crypto/.../batch=run/...` (by run time)
- **Batch (year/month/week/day)**: partitioned by **article issue date** under `news/transformed/crypto/year=.../month=.../day=...` etc.

Set `AWS_NEWS_BUCKET` or `AWS_DEFAULT_BUCKET` in `.env` for S3 uploads. For agentic mode, set `OPENAI_API_KEY` or `ANTHROPIC_API_KEY` and optionally `LLM_PROVIDER` (default: openai).

In [1]:
import sys
from pathlib import Path

# Resolve project root: run from repo root or notebooks/etl/
_cwd = Path(".").resolve()
project_root = _cwd if (_cwd / "src").is_dir() else (_cwd.parent.parent if _cwd.name == "etl" else _cwd)
src_path = project_root / "src"
if src_path.is_dir():
    sys.path.insert(0, str(project_root))
    sys.path.insert(0, str(src_path))
else:
    raise FileNotFoundError(f"Expected src at {src_path}. Run from repo root or notebooks/etl/.")

import pandas as pd

In [2]:
# Options: set to True to skip VADER and use only agentic (LLM) enrichment
USE_AGENTIC_ONLY = True
SINCE = "2026-05-26"
UNTIL = "2026-06-03"
# For agentic-only: limit rows (None = no limit; set e.g. 10 for a quick test)
AGENTIC_MAX_ROWS = None
# Save every N rows (None = save all at end; 31 or 50 = save after each chunk for resilience)
AGENTIC_SAVE_EVERY_N_ROWS = None
# S3: per-article and/or batch (same for both modes)
UPLOAD_S3_PER_ARTICLE = True
UPLOAD_S3_BATCH = ["day"] #["run", "week", "month", "year", "day"]

In [3]:
# Run news ETL: either agentic-only (skip VADER) or VADER transform; then save to Postgres and S3.
# Agentic: saves all transformed rows (including any with llm_error). Query llm_error column later to retry/fix.

from datetime import datetime
from pipelines.etl_transform import (
    run_news_etl,
    save_transformed_news_to_postgres,
    _ensure_news_transformed_table,
    agentic_result_has_failures,
    build_s3_key_news_per_article,
    upload_news_batches_to_s3,
    upload_dataframe_to_s3_key,
)
from config.settings import get_settings
from storage.postgres.pgConn import PgConn
from storage.postgres import PostgresSQL_table_queries as q

if USE_AGENTIC_ONLY:
    from pipelines.etl_cli import ingest_news
    from agents.registry import get_llm_client
    from agents.transforms.agentic_transform import AgenticTextEnricher, FinancialMetricsTask
    from storage.cloud.CloudStorage import CloudStorageProvider

    df = ingest_news(since=SINCE, until=UNTIL)
    if df is None or df.empty:
        transformed_df = pd.DataFrame()
        print("No news data to process")
    else:
        # Show daterange of rows being transformed
        if "datetime" in df.columns:
            dt = pd.to_datetime(df["datetime"], errors="coerce")
            valid = dt.notna()
            if valid.any():
                mn, mx = dt.loc[valid].min(), dt.loc[valid].max()
                daterange = f"{mn.date()} to {mx.date()}" if hasattr(mn, "date") else f"{str(mn)[:10]} to {str(mx)[:10]}"
                print(f"Transforming {len(df)} rows (daterange: {daterange})")
        settings = get_settings()
        provider = getattr(settings.agent, "provider", "openai")
        transformed_df = pd.DataFrame()
        try:
            client = get_llm_client(provider)
            enricher = AgenticTextEnricher(client=client, task=FinancialMetricsTask())
            to_process = min(len(df), AGENTIC_MAX_ROWS) if AGENTIC_MAX_ROWS is not None else len(df)
            chunk_size = AGENTIC_SAVE_EVERY_N_ROWS

            if chunk_size is None or chunk_size <= 0:
                # Enrich all, then save and upload once at the end
                transformed_df = enricher.enrich_dataframe(df, max_rows=AGENTIC_MAX_ROWS)
                if not transformed_df.empty:
                    conn = PgConn(q.FINANCIAL_NEWS_TABLE_NAME)
                    _ensure_news_transformed_table(conn)
                    n = save_transformed_news_to_postgres(conn, transformed_df, agentic_enabled=True)
                    conn.close_connection()
                    print(f"Saved {n} rows to {q.FINANCIAL_NEWS_TRANSFORMED_TABLE_NAME}")
                    bucket = getattr(settings.aws, "news_bucket", None) or settings.aws.default_bucket
                    if (UPLOAD_S3_PER_ARTICLE or UPLOAD_S3_BATCH) and not bucket:
                        print(
                            "S3 upload skipped: set AWS_NEWS_BUCKET or AWS_DEFAULT_BUCKET in .env, "
                            "then restart Jupyter."
                        )
                    if bucket and (UPLOAD_S3_PER_ARTICLE or UPLOAD_S3_BATCH):
                        aws = CloudStorageProvider.AWS()
                        if UPLOAD_S3_PER_ARTICLE:
                            for _, row in transformed_df.iterrows():
                                aid = str(row.get("id", ""))
                                dt_str = row.get("datetime")
                                try:
                                    dt = pd.to_datetime(dt_str) if dt_str else datetime.utcnow()
                                except Exception:
                                    dt = datetime.utcnow()
                                key = build_s3_key_news_per_article(aid, dt, agentic=True)
                                upload_dataframe_to_s3_key(aws.s3_client, bucket, key, pd.DataFrame([row]))
                            print(f"Uploaded {len(transformed_df)} per-article CSVs to s3://{bucket}/")
                        if UPLOAD_S3_BATCH:
                            upload_news_batches_to_s3(aws.s3_client, bucket, transformed_df, UPLOAD_S3_BATCH, agentic=True)
                            print(f"Uploaded batches {UPLOAD_S3_BATCH} to s3://{bucket}/")
            else:
                # Process and save in chunks (resilient: partial progress persisted if run stops)
                conn = PgConn(q.FINANCIAL_NEWS_TABLE_NAME)
                _ensure_news_transformed_table(conn)
                bucket = getattr(settings.aws, "news_bucket", None) or settings.aws.default_bucket
                if (UPLOAD_S3_PER_ARTICLE or UPLOAD_S3_BATCH) and not bucket:
                    print(
                        "S3 upload skipped: set AWS_NEWS_BUCKET or AWS_DEFAULT_BUCKET in .env, "
                        "then restart Jupyter."
                    )
                aws = CloudStorageProvider.AWS() if bucket and (UPLOAD_S3_PER_ARTICLE or UPLOAD_S3_BATCH) else None
                total_saved = 0
                chunks_out = []
                for start in range(0, to_process, chunk_size):
                    end = min(start + chunk_size, to_process)
                    chunk_df = df.iloc[start:end].copy()
                    enriched = enricher.enrich_dataframe(chunk_df, max_rows=None)
                    if enriched.empty:
                        continue
                    n = save_transformed_news_to_postgres(conn, enriched, agentic_enabled=True)
                    total_saved += n
                    chunks_out.append(enriched)
                    if bucket and UPLOAD_S3_PER_ARTICLE:
                        for _, row in enriched.iterrows():
                            aid = str(row.get("id", ""))
                            dt_str = row.get("datetime")
                            try:
                                dt = pd.to_datetime(dt_str) if dt_str else datetime.utcnow()
                            except Exception:
                                dt = datetime.utcnow()
                            key = build_s3_key_news_per_article(aid, dt, agentic=True)
                            upload_dataframe_to_s3_key(aws.s3_client, bucket, key, pd.DataFrame([row]))
                    print(f"  Chunk {start}-{end}: saved {n} rows (total so far: {total_saved})")
                conn.close_connection()
                transformed_df = pd.concat(chunks_out, ignore_index=True) if chunks_out else pd.DataFrame()
                print(f"Saved {total_saved} rows to {q.FINANCIAL_NEWS_TRANSFORMED_TABLE_NAME}")
                if bucket:
                    if UPLOAD_S3_PER_ARTICLE:
                        print(f"Uploaded {total_saved} per-article CSVs to s3://{bucket}/")
                    if UPLOAD_S3_BATCH and not transformed_df.empty:
                        upload_news_batches_to_s3(aws.s3_client, bucket, transformed_df, UPLOAD_S3_BATCH, agentic=True)
                        print(f"Uploaded batches {UPLOAD_S3_BATCH} to s3://{bucket}/")

            # Report any per-row llm_errors (all rows were saved; query llm_error in DB to retry later)
            if not transformed_df.empty and agentic_result_has_failures(transformed_df):
                err_series = transformed_df["llm_error"].fillna("").astype(str).str.strip()
                failed = err_series != ""
                n_failed = int(failed.sum())
                print(f"Note: {n_failed} / {len(transformed_df)} rows have llm_error set (saved anyway; query llm_error to retry).")
                for msg, count in transformed_df.loc[failed, "llm_error"].value_counts().items():
                    print(f"  [{count}] {msg}")
        except (KeyError, ValueError) as e:
            print(f"Agentic skipped (LLM not configured): {e}")
        except Exception as e:
            print(f"Agentic failed: {e}")
            transformed_df = pd.DataFrame()

    print(f"Agentic: transformed {len(transformed_df)} articles")
else:
    transformed_df = run_news_etl(
        since=SINCE,
        until=UNTIL,
        news_bucket=None,
        save_to_postgres=True,
        upload_s3_per_article=UPLOAD_S3_PER_ARTICLE,
        upload_s3_batch=UPLOAD_S3_BATCH if UPLOAD_S3_BATCH else None,
        sentiment_backend="vader",
        extract_tickers=True,
    )
    print(f"VADER: transformed {len(transformed_df)} articles")

Connection to the database successful!
Table name set to: financial_news_241118
Connection closed.
Transforming 836 rows (daterange: 2026-05-26 to 2026-06-03)
Agentic enrichment: processing 836 rows (daterange: 2026-05-26 to 2026-06-03, progress every 41 rows)
  progress: 41/836 rows
  progress: 82/836 rows
  progress: 123/836 rows
  progress: 164/836 rows
  progress: 205/836 rows
  progress: 246/836 rows
  progress: 287/836 rows
  progress: 328/836 rows
  progress: 369/836 rows
  progress: 410/836 rows
  progress: 451/836 rows
  progress: 492/836 rows
  progress: 533/836 rows
  progress: 574/836 rows
  progress: 615/836 rows
  progress: 656/836 rows
  progress: 697/836 rows
  progress: 738/836 rows
  progress: 779/836 rows
  progress: 820/836 rows
  progress: 836/836 rows
Agentic enrichment done: 836 rows
Connection to the database successful!
Table name set to: financial_news_241118
Table name set to: financial_news_transformed
Table 'financial_news_transformed' already exists.
Table

In [4]:
# Inspect agentic llm_error (run this when you see "Agentic enrichment had errors...")
if not transformed_df.empty and "llm_error" in transformed_df.columns:
    err_series = transformed_df["llm_error"].fillna("").astype(str).str.strip()
    failed = err_series != ""
    n_failed = failed.sum()
    if n_failed:
        print(f"Rows with llm_error: {n_failed} / {len(transformed_df)}")
        print("\n--- Unique llm_error messages (count) ---")
        print(transformed_df.loc[failed, "llm_error"].value_counts().to_string())
        print("\n--- Sample rows with errors (id, headline, llm_error) ---")
        sample = transformed_df.loc[failed, ["id", "headline", "llm_error"]].head(10)
        for _, row in sample.iterrows():
            print(f"  id: {row['id']}")
            print(f"  headline: {(str(row['headline'])[:60])}...")
            print(f"  llm_error: {row['llm_error']}")
            print()
    else:
        print("No rows have llm_error set.")
else:
    print("No transformed_df or no llm_error column to inspect.")

No transformed_df or no llm_error column to inspect.


In [5]:
# Inspect transformed output
if not transformed_df.empty:
    display(transformed_df.head())
    print(transformed_df.columns.tolist())

,id,source,headline,href,summary,content,author,minsread,datetime,created_at,...,llm_immediacy,llm_impact_horizon,llm_confidence,llm_novelty_score,llm_sentiment_label,llm_impact_level,llm_signal,llm_actionable,llm_sectors,llm_key_facts
0,2962932069771145171,decrypt,$1.3B Worth of BlackRock's IBIT Changes Hands ...,https://finance.yahoo.com/markets/crypto/artic...,,A $1.3 billion block of BlackRock’s iShares Bi...,Akash Girimath,3 min read,2026-05-27 11:13:46,2026-05-28 03:03:24.569154,...,0.4,short_term,0.7,-0.1,negative,medium,bearish,True,[crypto],[$1.3 billion block of IBIT shares sold in dar...
1,2049398938979137369,BeInCrypto,$1.47 Million in Satoshi Era Bitcoin Moves Aft...,https://finance.yahoo.com/markets/crypto/artic...,,A Satoshi-era Bitcoin wallet that sat untouche...,Lockridge Okoth,2 min read,2026-05-31 22:00:00,2026-06-01 03:05:55.154594,...,0.3,short_term,0.6,0.2,neutral,low,neutral,False,[crypto],"[20 BTC moved after 15.8 years of dormancy, Tr..."
2,3625388008948370077,BeInCrypto,$23 Billion EU Crypto Tax Forecast Draws Pushb...,https://finance.yahoo.com/markets/crypto/artic...,,"Patrick Hansen, Circle's EU strategy and polic...",Lockridge Okoth,2 min read,2026-05-30 17:39:18,2026-05-31 03:07:45.268124,...,0.4,medium_term,0.7,0.2,negative,medium,bearish,True,"[crypto, regulation]",[EU projects $23 billion in crypto tax revenue...
5,1906892630721503940,BeInCrypto,$7.5 Billion in Bitcoin and Ethereum Options E...,https://finance.yahoo.com/markets/options/arti...,,The crypto derivatives market faces a monthly ...,Luis Blanco,3 min read,2026-05-29 08:02:14,2026-05-30 03:29:55.887064,...,0.8,short_term,0.7,-0.1,negative,high,bearish,True,[crypto],"[$7.5 billion in options expire today, Bitcoin..."
7,1583750069504694230,CCN,"'BTC Ignites June Rally, XRP Set to Shock Mark...",https://finance.yahoo.com/markets/crypto/artic...,,"Key Takeaways\nYoungHoon Kim, known online as ...",Giuseppe Ciccomascolo,5 min read,2026-05-26 10:11:18,2026-05-27 03:17:39.710876,...,0.7,short_term,0.8,0.5,positive,medium,bullish,True,[crypto],"[Bitcoin is trading around $77,278, down by 11..."


['id', 'source', 'headline', 'href', 'summary', 'content', 'author', 'minsread', 'datetime', 'created_at', 'llm_financial_metrics', 'llm_entities', 'llm_ticker', 'llm_event_type', 'llm_overall_sentiment', 'llm_forward_sentiment', 'llm_surprise_score', 'llm_risk_score', 'llm_uncertainty_score', 'llm_impact_strength', 'llm_immediacy', 'llm_impact_horizon', 'llm_confidence', 'llm_novelty_score', 'llm_sentiment_label', 'llm_impact_level', 'llm_signal', 'llm_actionable', 'llm_sectors', 'llm_key_facts']


## Optional: Per-article only (no batch)
Use when you want only one CSV per article in S3.

In [6]:
# transformed_per_article = run_news_etl(
#     since="2026-01-01",
#     until="2026-01-28",
#     save_to_postgres=True,
#     upload_s3_per_article=True,
#     upload_s3_batch=None,
# )

## Optional: Batch only (no per-article)
Use when you want a single CSV per run, or per week/month/year.

In [7]:
# transformed_batch = run_news_etl(
#     date="2026-01-27",
#     save_to_postgres=True,
#     upload_s3_per_article=False,
#     upload_s3_batch=["run", "month"],
# )